# GenAI Assignment 1 — Colab runner
Run top to bottom. **Runtime → Change runtime type → T4 GPU** first.
Everything is backed up to Google Drive after each step, so if Colab disconnects just re-run the setup cells (1–5) and continue from where you stopped (Optuna studies resume automatically).

## 1. GPU check

In [ ]:
!nvidia-smi -L

## 2. Mount Drive (backup location)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BACKUP = '/content/drive/MyDrive/genai_a1'
!mkdir -p {BACKUP}

## 3. Get the code
If `REPO_URL` is empty, an upload button appears: choose `GenAI_Assignment1.zip`.

In [ ]:
REPO_URL = ''   # e.g. 'https://github.com/<you>/GenAI_Assignment1.git'
import os
if REPO_URL:
    if not os.path.exists('/content/repo'):
        !git clone {REPO_URL} /content/repo
    else:
        !cd /content/repo && git pull
elif not os.path.exists('/content/repo/genai'):
    from google.colab import files
    up = files.upload()
    name = list(up)[0]
    !mkdir -p /content/repo && unzip -oq "/content/{name}" -d /content/repo
%cd /content/repo
!ls

## 4. Install dependencies

In [ ]:
!pip install -q pytorch-msssim==1.0.0 optuna mlflow onnx onnxruntime

## 5. Paths + restore previous backup (safe to re-run)

In [ ]:
import os
os.environ['GENAI_DATA'] = '/content/data'
os.environ['GENAI_OUT'] = '/content/outputs'
!mkdir -p /content/data /content/outputs
# restore anything saved earlier
!test -d {BACKUP}/outputs && rsync -a {BACKUP}/outputs/ /content/outputs/ || true
!test -f {BACKUP}/pets_128.npz && cp -n {BACKUP}/pets_128.npz /content/data/ || true
!test -d {BACKUP}/manifests && rsync -a {BACKUP}/manifests/ /content/repo/manifests/ || true
!test -d {BACKUP}/models_onnx && rsync -a {BACKUP}/models_onnx/ /content/repo/models_onnx/ || true

def backup():
    get_ipython().system(f'rsync -a /content/outputs/ {BACKUP}/outputs/')
    get_ipython().system(f'rsync -a /content/repo/manifests/ {BACKUP}/manifests/')
    get_ipython().system(f'mkdir -p {BACKUP}/models_onnx && rsync -a /content/repo/models_onnx/ {BACKUP}/models_onnx/')
    get_ipython().system(f'cp -n /content/data/pets_128.npz {BACKUP}/ 2>/dev/null || true')
    print('backed up to', BACKUP)

## 6. Data: download, resize, split (seed 42), manifests
Downloads ~800 MB the first time; afterwards uses the cached 128×128 arrays.

In [ ]:
!python scripts/prepare_data.py
backup()

In [ ]:
!python -m pytest tests -q

---
# Task 1 — Universal denoising autoencoder

### 1a. Optuna search (~20–30 min on T4). Increase `--trials` if you have time.

In [ ]:
!python scripts/task1_optuna.py --trials 16 --epochs 6
backup()

### 1b. Retrain best config on the full schedule

In [ ]:
!python scripts/task1_train.py --epochs 40
backup()

### 1c. Test-set evaluation + figures, ONNX export + consistency check

In [ ]:
!python scripts/task1_eval.py
!python scripts/task1_export.py
backup()

### 1d. (Optional, for the report) limited-skip ablation

In [ ]:
!python scripts/task1_train.py --epochs 25 --skip --tag skip_ablation
!python scripts/task1_eval.py --tag skip_ablation
backup()

### Look at the results

In [ ]:
from IPython.display import Image, display
import pandas as pd
R = '/content/outputs/results/task1'
display(pd.read_csv(f'{R}/test_final_by_type.csv'))
for f in ['curves_final.png', 'optuna_history.png', 'optuna_importance.png', 'psnr_by_severity_final.png',
          'examples_a_final.png', 'examples_b_final.png', 'failures_final.png']:
    print(f); display(Image(f'{R}/{f}', width=700))